Módulo EBAC - Visualização Avançada (Dash)

In [54]:

df = pd.read_csv('ecommerce_estatistica.csv')

print(df.head())
print(df.info())

   Unnamed: 0                                             Título  Nota  \
0           1  Kit 10 Cuecas Boxer Lupo Cueca Box Algodão Mas...   4.5   
1           2  Kit Com 10 Cuecas Boxer Algodão Sem Costura Zo...   4.7   
2           3  Kit 10 Cuecas Boxer Mash Algodão Cotton Box Or...   4.6   
3           4  Kit 3 Short Jeans Feminino Cintura Alta Barato...   4.4   
4           5  Blusa + Calça Térmica Treino Futebol Criança I...   4.7   

   N_Avaliações  Desconto            Marca         Material  \
0        3034.0      18.0             lupo          algodão   
1        5682.0      20.0            zorba          algodão   
2        1700.0      22.0             mash          algodão   
3         507.0       9.0     menina linda             jean   
4          58.0       5.0  roupa zero grau  termico unissex   

                Gênero        Temporada  \
0            Masculino   outono/inverno   
1            Masculino     não definido   
2            Masculino  primavera/verão   
3   

Gráficos

In [64]:
import plotly.express as px
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf
from dash import Dash, html, dcc

df = pd.read_csv('ecommerce_estatistica.csv')

def gera_todos_os_graficos(df):

  # Histrograma
  fig1 = px.histogram(
      df,x='Preço',nbins=100, title='Histograma - Distribuição de Preço dos Produtos')
  fig1.update_traces(marker_color='#42f5ad', opacity=0.8)
  fig1.update_layout(
      xaxis_title='Preço(R$)',yaxis_title='Qtd. de Produtos', width=900, height=500)


  # Gráfico de Dispersão
  fig2 = px.scatter(
      df, x='Preço', y='Qtd_Vendidos_Cod', color='Gênero',marginal_x='histogram', marginal_y='histogram',
      title='Relação entre Preço e Quantidade Vendida por Gênero')


  # Mapa de Calor
  corr = df[['Preço', 'Desconto_MinMax']].corr()

  fig3 = px.imshow(
      corr, text_auto='.2f', color_continuous_scale='RdBu_r', zmin=-1, zmax=1,
      title='Matriz de Correlação: Preço e Desconto')

  fig3.update_traces(
      texttemplate='%{z:.2f}',
      textfont_size=14)


  # Gráfico de Barra
  df_material_media = df.groupby('Material', as_index=False)['Preço'].mean()

  fig4 = px.bar(
      df_material_media, x='Material', y='Preço', color='Material',color_continuous_scale='Viridis',
      title='Preço Médio dos Produtos por Material', labels={'Preço':'Preço Médio (R$)','Material': 'Material'})
  fig4.update_layout(
      xaxis_tickangle=-45,xaxis_title='Material',yaxis_title='Preço Médio (R$)', showlegend=False)


  # Gráfico de Pizza
  cores_customizadas = ['#e7f525', '#eb790e', '#c2108f', '#19046b', '#19c2ab', '#07a607', '#6311a6']
  contagem_temporada = df['Temporada'].value_counts().reset_index()
  contagem_temporada.columns = ['Temporada', 'Quantidade']

  fig5 = px.pie(
      contagem_temporada, names='Temporada', values='Quantidade',
      title='Distribuição Percentual de Produtos por Temporada', color_discrete_sequence=cores_customizadas)
  fig5.update_traces(
      textinfo='percent+label', insidetextorientation='radial', pull=[0.05] * len(contagem_temporada))
  fig5.update_layout(width=600, height=600)


  # Gráfico de Densidade
  fig6 = px.violin(
      df, x='Preço', color='Temporada', orientation='h', points=False, color_discrete_sequence=px.colors.qualitative.Set2,
      title='Comparação da Densidade da Preçopor Temporada')


  # Gráfico de Regressão
  fig7 = px.scatter(
      df, x='N_Avaliações', y='Preço', trendline='ols',trendline_color_override='#700629',
      title='Relação entre Número de Avaliações e Preço', labels={'N_Avaliações': 'Número de Avaliações', 'Preço':'Preço (R$)'})
  fig7.update_traces(
      marker=dict(color='#351640', opacity=0.5, size=8), selector=dict(mode='markers'))
  fig7.update_layout(
      xaxis_title='Número de Avaliações', yaxis_title='Preço (R$)',
      xaxis=dict(showgrid=True,gridcolor='rgba(0,0,0,0.1)',gridwidth=1),
      yaxis=dict(showgrid=True,gridcolor='rgba(0,0,0,0.1)',gridwidth=1))


  return fig1, fig2, fig3,fig4, fig5, fig6, fig7


# App
def cria_app(df):
  app= Dash(__name__)

  fig1, fig2, fig3,fig4, fig5, fig6, fig7 = gera_todos_os_graficos(df)

  fig1.show(),
  fig2.show(),
  fig3.show(),
  fig4.show(),
  fig5.show(),
  fig6.show(),
  fig7.show()

  app.layout = html.Div([
      dcc.Graph(figure=fig1),
      dcc.Graph(figure=fig2),
      dcc.Graph(figure=fig3),
      dcc.Graph(figure=fig4),
      dcc.Graph(figure=fig5),
      dcc.Graph(figure=fig6),
      dcc.Graph(figure=fig7)])
  return app

# Executar App
if __name__ == '__main__':
  app = cria_app(df)

  app.run(debug=True, port=8050)


<IPython.core.display.Javascript object>